In [9]:
# train_distilbert_complexity.py
import os
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import torch

# ---- Config ----
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./distilbert_complexity"
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
MAX_LEN = 128
THRESHOLDS = [0.33, 0.66]   # map continuous complexity -> classes
LABELS = ["simple", "medium", "conceptual"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE)

# ---- 1) Load dataset ----
ds = load_dataset("wesley7137/question_complexity_classification")
# dataset has 'question' and 'complexity' (float 0..1)
print("Splits:", ds.keys())

# If there's no validation split, create one
if "validation" not in ds:
    ds = ds["train"].train_test_split(test_size=0.1)
    ds["validation"] = ds["test"]

# ---- 2) Map continuous target to discrete labels ----
def to_label(score):
    if score < THRESHOLDS[0]:
        return 0
    elif score < THRESHOLDS[1]:
        return 1
    else:
        return 2

def map_to_label(example):
    example["label"] = to_label(example["complexity"])
    return example

ds = ds.map(map_to_label)

# ---- 3) Tokenizer & preprocessing ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    toks = tokenizer(batch["question"], truncation=True, padding=False, max_length=MAX_LEN)
    toks["labels"] = batch["label"]
    return toks

tokenized = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)

# Set format to torch
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# ---- 4) Model ----
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS))
model.to(DEVICE)

# ---- 5) Metrics ----
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1m = f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1m}

# ---- 6) Trainer ----
data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ---- 7) Train ----
trainer.train()

# ---- 8) Save ----
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved model to", OUTPUT_DIR)

# ---- 9) Quick inference helper ----
import torch.nn.functional as F
def predict(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(**enc)
        logits = out.logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
    return [{"text": t, "pred_label": LABELS[p], "probs": list(prob)} for t, p, prob in zip(texts, preds, probs)]

# Example
if __name__ == "__main__":
    examples = [
        "What is photosynthesis?",
        "Explain the chain of events and long-term political consequences leading up to World War I."
    ]
    print(predict(examples))


Device: cpu


Repo card metadata block was not found. Setting CardData to empty.


Splits: dict_keys(['train'])


Map:   0%|          | 0/12643 [00:00<?, ? examples/s]


KeyError: 'complexity'